In [1]:
import os
os.chdir("/Users/trentonsmiley/NCAA/backend")

import pandas as pd
import numpy as np
import unicodedata
import re

In [2]:
import requests 
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import unicodedata
import re

ModuleNotFoundError: No module named 'requests'

In [3]:
SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "v"}

def clean_name(name):
    """Removes nicknames, normalizes foreign characters, and cleans up formatting."""
    name = re.sub(r'"[^"]+"', '', name).strip()  # remove nicknames
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode()
    name = re.sub(r"[^A-Za-z0-9\- ]", '', name).strip()
    name = re.sub(r"\s+", " ", name)
    return name


def format_player_url(player):
    base_url = (
        "https://www.sports-reference.com/req/202511211/cbb/images/players/"
        "{}-{}{}-1.jpg"
    )

    cleaned = clean_name(player)
    parts = cleaned.split(" ")

    if len(parts) < 2:
        return None

    first = parts[0].lower()
    last = parts[1].lower()  # assume FORMAT: first last

    suffix = ""

    # CASE 1: Name has 3+ parts → could be middle OR suffix
    if len(parts) >= 3:
        possible_suffix = parts[-1].lower().replace(".", "")

        if possible_suffix in SUFFIXES:
            # Correct: Jeremy Fears Jr → jeremy-fears-jr
            suffix = "-" + possible_suffix
        else:
            # Middle initial name: John A Smith → john-smith-a
            middle = parts[1].replace(".", "").lower()
            last = parts[-1].lower()      # last token is last name
            suffix = "-" + middle if middle else ""

    return base_url.format(first, last, suffix)


# Load players with team info
players_df = pd.read_csv("data/players_box.csv")
conferences_df = pd.read_csv("data/conferences.csv")

# Get D1 teams and drop duplicates by player+team to get unique combinations
d1_teams = conferences_df[conferences_df.Conference != "Not D1"]["Team"].unique()
unique_players = players_df[players_df.Team.isin(d1_teams)].drop_duplicates(["Player", "Team"])

data = []
for _, row in unique_players.iterrows():
    player_name = row["Player"]
    team = row["Team"]
    # Create composite key format for uniqueness
    composite_key = f"{player_name}|{team}"
    link = format_player_url(player_name)
    data.append({"Player": composite_key, "Link": link})

result_df = pd.DataFrame(data)
result_df.to_csv("data/player_photos.csv", index=False)
print(f"Generated {len(result_df)} player photo entries")
print("\nSample entries (including Brandon Benjamin):")
print(result_df[result_df["Player"].str.contains("Brandon Benjamin", na=False)].to_string())

Generated 5241 player photo entries

Sample entries (including Brandon Benjamin):
                          Player                                                                                      Link
923   Brandon Benjamin|Fairfield  https://www.sports-reference.com/req/202511211/cbb/images/players/brandon-benjamin-1.jpg
4009  Brandon Benjamin|San Diego  https://www.sports-reference.com/req/202511211/cbb/images/players/brandon-benjamin-1.jpg
